# Speculative Decoding — Qwen2 1.5B × GSM8K

**What this does:**
Runs speculative decoding using HuggingFace's built-in `assistant_model`
parameter which handles token-level draft/verify automatically.

**Two configurations run in one session:**
- Config 1: W4A16 draft → W8A16 verifier
- Config 2: W4A16 draft → BF16 verifier

**What speculative decoding actually does:**
The draft model (W4A16) proposes K tokens at a time.
The verifier model checks each proposed token and either accepts it
or rejects it and substitutes a corrected token.
This is different from self-consistency — it intervenes DURING generation
rather than after, catching errors before they cascade.

**Metrics recorded per problem:**
- Final answer accuracy
- Generation time vs W4A16 greedy baseline

**Dataset:** GSM8K, same 50 problems as all other experiments

**Estimated time:** ~2.5 hours for both configs combined

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
!pip install -q "transformers>=4.40.0" "datasets>=2.19.0" "accelerate>=0.29.0" "bitsandbytes>=0.43.0"
print('Done')

In [ ]:
# ── CELL 2: GPU check ─────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime > Change runtime type > T4 GPU')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── CELL 3: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/quantization_research/speculative_decoding'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── CELL 4: Imports ───────────────────────────────────────────────────────────
import re, gc, random, time, os
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print('Imports OK')

In [ ]:
# ── CELL 5: Config ────────────────────────────────────────────────────────────

MODEL_ID       = 'Qwen/Qwen2-1.5B-Instruct'
SEED           = 42
MAX_NEW_TOKENS = 1024

TEST_IDS = {
    'easy':   [473, 624, 549, 1028, 958, 263, 493, 885, 1103, 450,
                591, 235, 105, 722, 1235, 433, 997, 29, 728, 1035],
    'medium': [1089, 359, 872, 209, 850, 457, 718, 61, 267, 187,
                586, 438, 1310, 858, 775, 1041, 227, 1202, 1044, 326],
    'hard':   [676, 1314, 361, 721, 629, 814, 175, 465, 199, 830],
}
ALL_TEST_IDS = TEST_IDS['easy'] + TEST_IDS['medium'] + TEST_IDS['hard']
print(f'Test set: {len(ALL_TEST_IDS)} problems')

In [ ]:
# ── CELL 6: Load GSM8K test problems ──────────────────────────────────────────
print('Loading GSM8K test split...')
ds_test = load_dataset('openai/gsm8k', 'main', split='test')

id_set = set(ALL_TEST_IDS)
test_problems = []
for idx, row in enumerate(ds_test):
    if idx in id_set:
        if idx in TEST_IDS['easy']:     diff = 'easy'
        elif idx in TEST_IDS['medium']: diff = 'medium'
        else:                           diff = 'hard'
        test_problems.append({
            'problem_id': idx, 'difficulty': diff,
            'question': row['question'], 'answer': row['answer'],
        })

print(f'Loaded {len(test_problems)} problems')

In [ ]:
# ── CELL 7: Answer extraction + prompt formatting ─────────────────────────────

def extract_ground_truth(answer_str):
    match = re.search(r'####\s*([\d,\.\-]+)', answer_str)
    if match: return match.group(1).replace(',', '').strip()
    return None

def extract_prediction(output_str):
    m = re.search(r'\\boxed\{([\d,\.\-]+)\}', output_str)
    if m: return m.group(1).replace(',', '').strip()
    m = re.search(r'####\s*([\d,\.\-]+)', output_str)
    if m: return m.group(1).replace(',', '').strip()
    nums = re.findall(r'\b([\d,\.]+)\b', output_str)
    if nums: return nums[-1].replace(',', '').strip()
    return None

def format_prompt(question):
    return (
        '<|im_start|>system\n'
        'You are a math problem solver. Show your reasoning step by step. '
        'End your answer with the final number in \\boxed{}.\n'
        '<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

print('Helpers ready')

In [ ]:
# ── CELL 8: Load tokenizer ────────────────────────────────────────────────────
# One tokenizer shared by both models

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'
print('Tokenizer ready')

In [ ]:
# ── CELL 9: Run speculative decoding ──────────────────────────────────────────


configs = [
    {
        'name':         'W4A16_draft_W8A16_verify',
        'draft_quant':  BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.bfloat16),
        'verify_quant': BitsAndBytesConfig(load_in_8bit=True),
        'output_file':  os.path.join(OUTPUT_DIR, 'spec_dec_w4a16_w8a16.csv'),
    },
    {
        'name':         'W4A16_draft_BF16_verify',
        'draft_quant':  BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.bfloat16),
        'verify_quant': None,   # BF16 — no quantization
        'output_file':  os.path.join(OUTPUT_DIR, 'spec_dec_w4a16_bf16.csv'),
    },
]

all_results = []

for cfg in configs:
    print('='*60)
    print(f'Config: {cfg["name"]}')
    print('='*60)

    # Load draft model (W4A16)
    print('Loading draft model (W4A16)...')
    draft_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=cfg['draft_quant'],
        device_map='auto', trust_remote_code=True,
    )
    draft_model.eval()
    print(f'  Draft GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB')

    # Load verifier model
    print('Loading verifier model...')
    if cfg['verify_quant'] is not None:
        verify_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=cfg['verify_quant'],
            device_map='auto', trust_remote_code=True,
        )
    else:
        verify_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16,
            device_map='auto', trust_remote_code=True,
        )
    verify_model.eval()
    print(f'  Total GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB')

    
    existing = pd.read_csv(cfg['output_file']) if os.path.exists(cfg['output_file']) \
               else pd.DataFrame()
    done_ids = set(existing['problem_id'].tolist()) if len(existing) > 0 else set()

    results = []
    t0 = time.time()

    for i, prob in enumerate(test_problems):
        if prob['problem_id'] in done_ids:
            print(f'  [{i+1}/50] pid={prob["problem_id"]} skipping')
            continue

        gt     = extract_ground_truth(prob['answer'])
        prompt = format_prompt(prob['question'])
        inputs = tokenizer(prompt, return_tensors='pt').to(verify_model.device)

        # Time the speculative decoding generation
        t_start = time.time()
        with torch.no_grad():
            # HuggingFace built-in speculative decoding
            # assistant_model = draft model that proposes tokens
            # main model (verify_model) accepts or rejects each token
            output_ids = verify_model.generate(
                **inputs,
                assistant_model  = draft_model,
                max_new_tokens   = MAX_NEW_TOKENS,
                do_sample        = False,
                pad_token_id     = tokenizer.eos_token_id,
            )
        t_end = time.time()
        gen_time = t_end - t_start

        generated = tokenizer.decode(
            output_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        pred    = extract_prediction(generated)
        correct = (pred is not None and gt is not None
                   and pred.strip() == gt.strip())
        elapsed = (time.time() - t0) / 60

        results.append({
            'config':       cfg['name'],
            'draft':        'W4A16',
            'verifier':     'W8A16' if cfg['verify_quant'] is not None
                            and 'load_in_8bit' in str(cfg['verify_quant']) else 'BF16',
            'problem_id':   prob['problem_id'],
            'difficulty':   prob['difficulty'],
            'ground_truth': gt,
            'predicted':    pred,
            'correct':      correct,
            'gen_time_s':   round(gen_time, 2),
            'raw_output':   generated[:500],
        })

        # Save after every problem
        all_rows = pd.concat([existing, pd.DataFrame(results)], ignore_index=True)
        all_rows.to_csv(cfg['output_file'], index=False)

        status = 'CORRECT' if correct else 'wrong'
        print(f'  [{i+1}/50] pid={prob["problem_id"]:<5} '
              f'{prob["difficulty"]:<8} '
              f'pred={str(pred):<12} gt={str(gt):<10} '
              f'{status}  {gen_time:.1f}s  {elapsed:.0f}min')

    # Summary for this config
    n_correct = sum(r['correct'] for r in results)
    print(f'Config done: {n_correct}/{len(results)} '
          f'({100*n_correct/max(len(results),1):.1f}%)  '
          f'{(time.time()-t0)/60:.1f} min')
    all_results.extend(results)

    # Free memory before loading next config
    del draft_model, verify_model
    gc.collect()
    torch.cuda.empty_cache()
    print(f'GPU mem after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    print()

In [ ]:
# ── CELL 10: Full summary ──────────────────────────────────────────────────────
# Compare accuracy and timing across both configs

print('='*60)
print('SPECULATIVE DECODING RESULTS SUMMARY')
print('='*60)

# Load both output files
for fname, label in [
    (os.path.join(OUTPUT_DIR, 'spec_dec_w4a16_w8a16.csv'), 'W4A16 draft + W8A16 verifier'),
    (os.path.join(OUTPUT_DIR, 'spec_dec_w4a16_bf16.csv'),  'W4A16 draft + BF16 verifier'),
]:
    if not os.path.exists(fname):
        print(f'{label}: not yet available')
        continue
    df = pd.read_csv(fname)
    acc = df['correct'].mean()*100
    mean_time = df['gen_time_s'].mean()
    print(f'{label}:')
    print(f'  Accuracy:  {acc:.1f}% ({int(df["correct"].sum())}/{len(df)})')
    for diff in ['easy','medium','hard']:
        d = df[df['difficulty']==diff]
        print(f'  {diff}: {d["correct"].mean()*100:.1f}% ({int(d["correct"].sum())}/{len(d)})')
    print(f'  Mean generation time: {mean_time:.1f}s per problem')
    print()

print('Reference (from Shaswat):')
print('  W4A16 Greedy:           50.0%')
print('  W4A16 SC N=5:           38.0%  (SC hurts at W4A16)')
print('  W8A16 SC N=5:           52.0%  (best inference-time)')

In [ ]:
# ── CELL 11: Save combined results ────────────────────────────────────────────
combined_path = os.path.join(OUTPUT_DIR, 'spec_dec_combined.csv')
df1 = pd.read_csv(os.path.join(OUTPUT_DIR, 'spec_dec_w4a16_w8a16.csv'))
df2 = pd.read_csv(os.path.join(OUTPUT_DIR, 'spec_dec_w4a16_bf16.csv'))
combined = pd.concat([df1, df2], ignore_index=True)
combined.to_csv(combined_path, index=False)
print(f'Combined results saved: {combined_path}')
print(f'Total rows: {len(combined)}')